# Using the Assistant Axis pipeline from the original paper


We are using the code provided by (https://github.com/safety-research/assistant-axis) to generate responses, extract activations, generate judge scores, calculate role vectors and the Assistant Axis.


## Running Assistant Axis Extraction Pipeline for Qwen3-32B

Running the pipeline on the 32B model in English using a reduced dataset (both roles and extraction questions were reduced).

### Set up code

In [ ]:
# Install the uv package
!pip install uv

# Install require libaries
import os
from pathlib import Path
from google.colab import userdata

In [ ]:
# Clone the Assistant Axis github repo in google colab root directory
!git clone https://github.com/safety-research/assistant-axis.git

# Cd into assistant-axis directory and create root directory
%cd /content/assistant-axis
ROOT_DIR = Path.cwd()

# Install environment variables using the uv package
!uv sync

# Clone the SAIN Utrecht Summer Challenge github repo in google colab
!git clone https://github.com/Arcee183/SAIN_Utrecht_Summer_Challenge.git

# Directories for roles and pipeline
AXIS_DIR = ROOT_DIR / "data"
ROLES_DIR = AXIS_DIR / "roles"
PIPELINE_DIR = ROOT_DIR / "pipeline"

# Main directories for storing data
SAIN_DIR = ROOT_DIR / "SAIN_Utrecht_Summer_Challenge"
DATA_DIR = SAIN_DIR / "data" / "outputs" / "qwen3-32B" / "English"

In [ ]:
# Save the api key to the environment variable to run the judge pipeline
os.environ["OPENAI_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

### Step 1: Generate responses from the model
The following code block will generate responses from the model that's selected in the model flag and automatically store the results in the designated output directory (output_dir flag)

In [ ]:
# Define our role list for which roles we are going to study
roles = [
    'accountant',
    'actor',
    'addict',
    'amnesiac',
    'angel',
    'bard',
    'blogger',
    'builder',
    'comedian',
    'competitor',
    'conservator',
    'critic',
    'demon',
    'default',
    'designer',
    'evaluator',
    'evangelist',
    'exile',
    'grader',
    'hybrid',
    'idealist',
    'leviathan',
    'librarian',
    'linguist',
    'marketer',
    'merchant',
    'narrator',
    'navigator',
    'networker',
    'optimist',
    'oracle',
    'organizer',
    'pirate',
    'podcaster',
    'predator',
    'presenter',
    'prey',
    'proofreader',
    'provincial',
    'provocateur',
    'researcher',
    'retiree',
    'robot',
    'sage',
    'scholar',
    'skeptic',
    'soldier',
    'spy',
    'strategist',
    'teacher',
    'theorist',
    'traditionalist',
    'trainer',
    'whale',
    'widow',
    'zeitgeist'
]

In [ ]:
# Use the uv package to run 1_generate.py file with the following flags:
!uv run {PIPELINE_DIR}/1_generate.py \
    --model Qwen/Qwen3-32B \
    --output_dir {DATA_DIR}/responses \
    --roles_dir {ROLES_DIR}/instructions \
    --questions_file {AXIS_DIR}/extraction_questions.jsonl \
    --question_count 50 \
    --roles {' '.join(roles)} # use this flag to test for a single role instead of full 275

### Step 2: Extract activations from the model to calculate role vectors
This code block will extract activations from the model given the responses. Activations from all layers can be extracted or a specific layer (change the layer flag for this option)

In [ ]:
# Use the uv package to run 2_activations.py with the following flags:
!uv run {PIPELINE_DIR}/2_activations.py \
    --model Qwen/Qwen3-32B \
    --responses_dir {DATA_DIR}/responses \
    --output_dir {DATA_DIR}/activations \
    --batch_size 16 \
    --roles {' '.join(roles)} \
    --layers "6,11,16,21,26,31,36,41,46,51,56,61"

### Step 3: Use GPT LLM Judge to classify responses from model
The judge will classify responses from model to filter out somewhat roleplaying responses and only use the full roleplaying ones to calculate the assistant-axis.

In [ ]:
# Use the uv package to run 3_judge.py with the following flags:
# Default judge model is gpt-4.1-mini
!uv run {PIPELINE_DIR}/3_judge.py \
    --responses_dir {DATA_DIR}/responses \
    --roles_dir {ROLES_DIR}/instructions \
    --roles {' '.join(roles)} \
    --output_dir {DATA_DIR}/scores

### Step 4: Calculate role vectors for each role
Each role will result in a role vector using the activations from step 2.

In [ ]:
# Use the uv package to run 4_vectors.py with the following flags:
!uv run {PIPELINE_DIR}/4_vectors.py \
    --activations_dir {DATA_DIR}/activations \
    --scores_dir  {DATA_DIR}/scores \
    --output_dir {DATA_DIR}/vectors \
    --min_count 10

### Step 5: Calculate the Assistant Axis using the contrast vector method
Using the specified contrast vector method from the paper, the final Assistant Axis is calculated and stored. This Axis can then be used to plot the cosine similarity with the original persona vectors using the notebook from the github repo (https://github.com/safety-research/assistant-axis/blob/master/notebooks/visualize_axis.ipynb).

In [ ]:
# Use the uv package to run 5_axis.py with the following flags:
!uv run {PIPELINE_DIR}/5_axis.py \
    --vectors_dir {DATA_DIR}/vectors \
    --output {DATA_DIR}/english_prompt_axis.pt